# Multimodal Biomedical Signal Classification
## Hybrid CNN–LSTM with Adaptive Normalization and XAI
**Thesis pipeline — run every cell top-to-bottom.**

Expected total runtime on a modern GPU: ~32 hours.  
On CPU only: ~3 hours for ECG-only fast-track (see Cell 4b).


## 0 · Required directory layout
Before running anything, ensure your files match this tree exactly.
```
thesis/                          ← this notebook lives here
├── thesis_pipeline.ipynb
├── main.py
├── preprocessing_pipeline.py
├── baseline_models.py
├── hybrid_model.py
├── ablation_framework.py
├── multimodal_fusion.py
├── xai_and_pipeline.py
├── evaluation_suite.py
├── verify_patches.py
├── config.yaml
├── requirements.txt
└── data/
    └── raw/
        ├── mit-bih/             ← all .dat / .hea / .atr files FLAT here
        │   ├── 100.dat
        │   ├── 100.hea
        │   ├── 100.atr
        │   └── ... (48 records × 3 files = 144 files total)
        ├── eeg-motor/           ← subject folders here
        │   ├── S001/
        │   │   ├── S001R01.edf
        │   │   └── ... (14 runs per subject)
        │   ├── S002/ ...
        │   └── S109/ ...
        └── ppg-dalia/           ← .pkl files FLAT here
            ├── S1.pkl
            ├── S2.pkl
            └── ... (S1.pkl – S15.pkl)
```
> **MIT-BIH**: downloaded zip extracts to a flat directory — place all files directly in `mit-bih/`.  
> **EEG Motor Movement**: downloaded zip extracts to `files/` with `S001/` subdirs — move the `S001/`…`S109/` folders directly into `eeg-motor/`.  
> **PPG-DaLiA**: downloaded zip extracts to `PPG_FieldStudy/S1/S1.pkl` etc. — copy each `S*.pkl` into `ppg-dalia/` (flatten one level).


## 1 · Install dependencies
Run once. Safe to re-run — pip skips already-installed packages.


In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'torch', 'torchvision',
    'wfdb',
    'mne>=1.6',
    'neurokit2',
    'h5py',
    'scikit-learn',
    'scipy',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'tqdm',
])
print('Done.')


## 2 · Verify environment and data layout


In [ ]:
import sys, os
from pathlib import Path

# ── Python and packages ──────────────────────────────────────────────
import torch, numpy, scipy, sklearn, h5py, mne, wfdb
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}  '
      f'(device count: {torch.cuda.device_count()})')
print(f'numpy   : {numpy.__version__}')
print(f'h5py    : {h5py.__version__}')
print(f'mne     : {mne.__version__}')
print()

# ── Data layout check ────────────────────────────────────────────────
THESIS_DIR = Path('.')           # adjust if notebook is elsewhere
RAW        = THESIS_DIR / 'data' / 'raw'

checks = {
    'MIT-BIH .hea files'  : len(list((RAW / 'mit-bih').glob('*.hea'))),
    'MIT-BIH .dat files'  : len(list((RAW / 'mit-bih').glob('*.dat'))),
    'MIT-BIH .atr files'  : len(list((RAW / 'mit-bih').glob('*.atr'))),
    'EEG subject dirs'    : len(list((RAW / 'eeg-motor').glob('S[0-9][0-9][0-9]'))),
    'PPG .pkl files'      : len(list((RAW / 'ppg-dalia').glob('S*.pkl'))),
}
for label, count in checks.items():
    status = '✓' if count > 0 else '✗  ← MISSING'
    print(f'  {status}  {label}: {count}')

expected = {'MIT-BIH .hea files': 48, 'EEG subject dirs': 109, 'PPG .pkl files': 15}
for k, exp in expected.items():
    if checks[k] < exp:
        print(f'\nWARNING: {k} — found {checks[k]}, expected {exp}.')
        print('  Check the directory layout in Cell 0.')


## 3 · Verify patches (no data needed)
Confirms the three bug fixes compile and behave correctly before touching real data.


In [ ]:
exec(open('verify_patches.py').read())


## 4a · Full pipeline — all three modalities  *(GPU recommended)*
Runs preprocessing → baselines → hybrid → ablations → XAI → statistical eval → late fusion.  
**Skip to Cell 4b** if you are on CPU only or want a faster first run.


In [ ]:
import sys, os
sys.path.insert(0, '.')
os.chdir('.')               # ensure working dir = thesis root

from xai_and_pipeline import UnifiedRunConfig, end_to_end_run

cfg = UnifiedRunConfig(
    raw_data_dir       = 'data/raw',
    processed_dir      = 'data/processed',
    output_dir         = 'outputs',
    modalities         = ['ecg', 'eeg', 'ppg'],
    preprocessing_norm = 'none',      # proposed system: LearnableNorm handles it
    model_norm         = 'learnable',
    epochs             = 100,
    batch_size         = 64,
    learning_rate      = 5e-4,
    patience           = 20,
    seed               = 42,
    num_workers        = 4,
    run_xai            = True,
    n_xai_samples      = 32,
    run_ablations      = True,
    ablation_runs      = 3,
)

end_to_end_run(cfg, force_preprocess=False)


## 4b · Fast-track — ECG only on CPU  *(~2–3 hours)*
Runs the full pipeline on ECG only with reduced epochs and no ablations.  
Use this to verify the pipeline end-to-end before committing to a full run.


In [ ]:
import sys, os
sys.path.insert(0, '.')
os.chdir('.')

from xai_and_pipeline import UnifiedRunConfig, end_to_end_run

cfg_fast = UnifiedRunConfig(
    raw_data_dir       = 'data/raw',
    processed_dir      = 'data/processed_fast',
    output_dir         = 'outputs_fast',
    modalities         = ['ecg'],          # ECG only
    preprocessing_norm = 'none',
    model_norm         = 'learnable',
    epochs             = 30,               # reduced for speed
    batch_size         = 128,
    learning_rate      = 5e-4,
    patience           = 10,
    seed               = 42,
    num_workers        = 0,                # 0 = no multiprocessing (safer on some OS)
    run_xai            = True,
    n_xai_samples      = 8,
    run_ablations      = False,            # skip ablations for speed
    ablation_runs      = 1,
)

end_to_end_run(cfg_fast, force_preprocess=False)


## 4c · Control system — z-score preprocessing, no model norm
Run this **after** 4a/4b completes. Results feed into the ablation comparison (A0 vs A1).


In [ ]:
from xai_and_pipeline import UnifiedRunConfig, end_to_end_run

cfg_ctrl = UnifiedRunConfig(
    raw_data_dir       = 'data/raw',
    processed_dir      = 'data/processed_control',
    output_dir         = 'outputs_control',
    modalities         = ['ecg', 'eeg', 'ppg'],
    preprocessing_norm = 'zscore',        # control: offline z-score
    model_norm         = 'none',          # no model-level norm
    epochs             = 100,
    batch_size         = 64,
    patience           = 20,
    seed               = 42,
    num_workers        = 4,
    run_xai            = False,           # XAI already done on proposed system
    run_ablations      = False,
    ablation_runs      = 1,
)

end_to_end_run(cfg_ctrl, force_preprocess=False)


## 5 · Deployment analysis
Latency profiling, model size, FLOPs, quantisation comparison.  
Run after 4a/4b. Loads the best hybrid checkpoint for each modality.


In [ ]:
import torch
from xai_and_pipeline import UnifiedRunConfig, build_loaders_from_hdf5
from hybrid_model import HybridCNNLSTM
from evaluation_suite import deployment_report

OUTPUT_DIR    = 'outputs'          # change to 'outputs_fast' for fast-track
PROCESSED_DIR = 'data/processed'
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

from hybrid_model import HybridConfig
hybrid_cfg = HybridConfig(
    adaptive_norm = 'learnable',
    processed_dir = PROCESSED_DIR,
    output_dir    = OUTPUT_DIR + '/hybrid',
)

for modality in ['ecg', 'eeg', 'ppg']:
    ckpt_path = f'{OUTPUT_DIR}/hybrid/{modality}/best_hybrid.pt'
    h5_path   = f'{PROCESSED_DIR}/{modality}.h5'
    import os
    if not os.path.exists(ckpt_path) or not os.path.exists(h5_path):
        print(f'Skipping {modality} — checkpoint or HDF5 not found.')
        continue

    loaders  = build_loaders_from_hdf5(PROCESSED_DIR, modality, 64, 4, DEVICE)
    ds       = loaders['train'].dataset
    model    = HybridCNNLSTM(ds.n_channels, ds.n_classes, hybrid_cfg).to(DEVICE)
    ckpt     = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])

    deploy_dir = f'{OUTPUT_DIR}/deployment/{modality}'
    os.makedirs(deploy_dir, exist_ok=True)
    deployment_report(model, loaders['test'], ds.n_channels,
                      ds.seq_len, deploy_dir, DEVICE)
    print(f'Deployment report for {modality.upper()} → {deploy_dir}')


## 6 · Inspect results
Loads every key output file and displays it inline.  
All outputs also exist as files in `outputs/` for copy-pasting into the thesis.


In [ ]:
import pandas as pd, json, os
from IPython.display import display, Image

OUTPUT_DIR = 'outputs'

# ── Final three-way comparison (CNN vs LSTM vs Hybrid) ───────────────
comparison_path = f'{OUTPUT_DIR}/hybrid/final_comparison.csv'
if os.path.exists(comparison_path):
    df = pd.read_csv(comparison_path)
    print('\n=== Final comparison: CNN / LSTM / Hybrid ===')
    display(df.round(4))

# ── Per-modality test metrics ─────────────────────────────────────────
for modality in ['ecg', 'eeg', 'ppg']:
    for model in ['cnn', 'lstm']:
        path = f'{OUTPUT_DIR}/baselines/{modality}/{model}/test_metrics.json'
        if os.path.exists(path):
            m = json.load(open(path))
            print(f'{modality.upper()} {model.upper()}: '
                  f'acc={m["accuracy"]:.4f}  '
                  f'macro_f1={m["macro_f1"]:.4f}  '
                  f'latency={m["latency_ms_per_sample"]:.3f}ms')
    path = f'{OUTPUT_DIR}/hybrid/{modality}/test_metrics.json'
    if os.path.exists(path):
        m = json.load(open(path))
        print(f'{modality.upper()} HYBRID: '
              f'acc={m["accuracy"]:.4f}  '
              f'macro_f1={m["macro_f1"]:.4f}  '
              f'latency={m["latency_ms_per_sample"]:.3f}ms')
    print()


In [ ]:
# ── Statistical summary (mean ± std across seeds) ────────────────────
for modality in ['ecg', 'eeg', 'ppg']:
    stat_path = f'{OUTPUT_DIR}/statistical/{modality}/statistical_summary.csv'
    if os.path.exists(stat_path):
        df = pd.read_csv(stat_path)
        print(f'\n=== {modality.upper()} statistical summary ===')
        cols = ['model'] + [c for c in df.columns
                             if 'mean' in c or 'std' in c]
        display(df[cols].round(4))

# ── Wilcoxon significance tests ───────────────────────────────────────
for modality in ['ecg', 'eeg', 'ppg']:
    sig_path = f'{OUTPUT_DIR}/statistical/{modality}/significance_tests.csv'
    if os.path.exists(sig_path):
        df = pd.read_csv(sig_path)
        print(f'\n=== {modality.upper()} Wilcoxon tests (macro F1) ===')
        display(df[['model_a','model_b','a_mean','b_mean',
                     'p_value','significant','cohens_d']].round(4))


In [ ]:
# ── Ablation results ─────────────────────────────────────────────────
for modality in ['ecg', 'eeg', 'ppg']:
    ab_path = f'{OUTPUT_DIR}/ablations/{modality}/ablation_results.csv'
    if os.path.exists(ab_path):
        df = pd.read_csv(ab_path)
        print(f'\n=== {modality.upper()} ablation results ===')
        display_cols = [
            'ablation','preprocess_norm','model_norm',
            'use_lstm','use_multiscale','use_se','use_gated_fusion',
            'accuracy_mean','accuracy_std','macro_f1_mean','macro_f1_std',
        ]
        display(df[[c for c in display_cols if c in df.columns]].round(4))


In [ ]:
# ── Key figures inline ───────────────────────────────────────────────
import matplotlib.pyplot as plt, matplotlib.image as mpimg

fig_paths = [
    (f'{OUTPUT_DIR}/hybrid/final_comparison.png',       'Final model comparison'),
    (f'{OUTPUT_DIR}/ablations/ecg/ablation_results.png','ECG ablation study'),
    (f'{OUTPUT_DIR}/statistical/ecg/violin_macro_f1.png','ECG violin plot (macro F1)'),
    (f'{OUTPUT_DIR}/xai/ecg/ecg_Integrated_Gradients_class_profiles.png',
     'ECG IG attribution profiles'),
    (f'{OUTPUT_DIR}/xai/ecg/ecg_LRP_(ε-rule)_class_profiles.png',
     'ECG LRP attribution profiles'),
    (f'{OUTPUT_DIR}/hybrid/ecg/confusion_matrix.png',   'ECG confusion matrix'),
]

for path, title in fig_paths:
    if os.path.exists(path):
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.imshow(mpimg.imread(path))
        ax.axis('off')
        ax.set_title(title, fontsize=11)
        plt.tight_layout()
        plt.show()
    else:
        print(f'Figure not yet produced: {path}')


In [ ]:
# ── LaTeX tables (ready to paste into thesis) ────────────────────────
for modality in ['ecg', 'eeg', 'ppg']:
    tex_path = f'{OUTPUT_DIR}/statistical/{modality}/results_table.tex'
    if os.path.exists(tex_path):
        print(f'\n% ── {modality.upper()} results table ──')
        print(open(tex_path).read())


In [ ]:
# ── Deployment summary ───────────────────────────────────────────────
for modality in ['ecg', 'eeg', 'ppg']:
    cmp_path = f'{OUTPUT_DIR}/deployment/{modality}/deployment_comparison.csv'
    if os.path.exists(cmp_path):
        df = pd.read_csv(cmp_path)
        print(f'\n=== {modality.upper()} deployment comparison ===')
        display(df.round(4))


In [ ]:
# ── Late fusion modality weights ─────────────────────────────────────
weights_path = f'{OUTPUT_DIR}/late_fusion/modality_weights.json'
if os.path.exists(weights_path):
    w = json.load(open(weights_path))
    print('\nLearned modality weights (late fusion):')
    for mod, val in w.items():
        print(f'  {mod.upper():6s}: {val:.4f}')


## 7 · Output manifest
All files produced by the pipeline and their thesis section.
| File | Thesis section |
|------|----------------|
| `outputs/hybrid/final_comparison.csv` | Chapter 4 — Results table |
| `outputs/hybrid/final_comparison.png` | Chapter 4 — Figure |
| `outputs/statistical/*/statistical_summary.csv` | Chapter 4 — Mean±std table |
| `outputs/statistical/*/significance_tests.csv` | Chapter 4 — Wilcoxon table |
| `outputs/statistical/*/results_table.tex` | Chapter 4 — LaTeX table |
| `outputs/statistical/*/violin_macro_f1.png` | Chapter 4 — Figure |
| `outputs/ablations/*/ablation_results.csv` | Chapter 4 — Ablation table |
| `outputs/ablations/*/ablation_results.png` | Chapter 4 — Figure |
| `outputs/xai/*/ecg_*_class_profiles.png` | Chapter 5 — XAI figures |
| `outputs/hybrid/*/confusion_matrix.png` | Chapter 4 — Figure |
| `outputs/deployment/*/deployment_comparison.csv` | Chapter 6 — Deployment |
| `outputs/deployment/*/latency_profile.png` | Chapter 6 — Figure |
| `outputs/late_fusion/modality_weights.json` | Chapter 4 — Multimodal |
| `run.log` | Appendix — full training log |
